# SeqTrainer Tutorial 05: End-to-end mini CNN regressor

This notebook mirrors Tutorial 04, but uses a **regression head** to predict promoter activity (`target`) directly.

## Goal

Demonstrate an end-to-end regression workflow with a short training run.

Increase `NUM_CYCLES` later for longer training.

In [1]:
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from seqtrainer.data.sbol import build_dataset_from_files
from seqtrainer.data.materialized import MaterializedDataset
from seqtrainer.transforms.dna import one_hot_encode, pad_or_trim

print("Imports loaded")

Imports loaded


## 1) Load SBOL data for promoter activity regression

In [3]:
base = Path("../../data/sbol_data")
files = sorted(base.glob("sample_design_*.xml"))[:40]
df = build_dataset_from_files(files)

if df.empty:
    raise RuntimeError("No rows were materialized from SBOL inputs.")

print(f"Rows: {len(df)}")
df[["sequence", "target"]].head()

Rows: 40


,sequence,target
0,GCAAATTTTGCACAAAAAATAGGCTTTAGTGATTTGTTTTTGTTCA...,52.010547
1,GTATCTGCCTCCGATTCTCTGCAGAAGCAGAAAGACATTGGATCGA...,0.565387
2,CTTCCAGGCGGGTGGGGTCAATGTCCATCAGGGCAATATGCGCCGT...,0.690511
3,ATCAAAAATGAAGCCGATAACGGCCTGCGCAACACGCGTGGCACCA...,0.696464
4,GATCGGGCCGGAAGCCGGACACCGCGCAGGTTGGTACAACCACTAT...,1.094582


## 2) DNA preprocessing (fixed length + one-hot)

In [4]:
SEQ_LEN = 120

fixed_sequences = [pad_or_trim(seq, length=SEQ_LEN) for seq in df["sequence"].tolist()]
X = one_hot_encode(fixed_sequences)  # [N, L, C]
y = df["target"].to_numpy(dtype=np.float32)

# Conv1d input: [N, C, L]
X_t = torch.tensor(np.transpose(X, (0, 2, 1)), dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

X_t.shape, y_t.shape

(torch.Size([40, 5, 120]), torch.Size([40, 1]))

## 3) Train/val/test split using `MaterializedDataset`

In [5]:
examples = [{"idx": i} for i in range(len(y))]
materialized = MaterializedDataset(examples, metadata={"tutorial": "cnn_regression_demo"})
train_ds, val_ds, test_ds = materialized.train_val_test_split(0.7, 0.15, 0.15, seed=42)

def to_index_tensor(split):
    return torch.tensor([row["idx"] for row in split.examples], dtype=torch.long)

train_idx, val_idx, test_idx = map(to_index_tensor, (train_ds, val_ds, test_ds))
len(train_idx), len(val_idx), len(test_idx)

(28, 6, 6)

## 4) Build dataloaders

In [6]:
BATCH_SIZE = 16

train_loader = DataLoader(TensorDataset(X_t[train_idx], y_t[train_idx]), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(TensorDataset(X_t[val_idx], y_t[val_idx]), batch_size=BATCH_SIZE)
test_loader = DataLoader(TensorDataset(X_t[test_idx], y_t[test_idx]), batch_size=BATCH_SIZE)

len(train_loader), len(val_loader), len(test_loader)

(2, 1, 1)

## 5) Define a compact CNN backbone + regression head

In [7]:
class TinyDNACNNRegressor(nn.Module):
    def __init__(self, channels: int = 5):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv1d(channels, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.AdaptiveMaxPool1d(1),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.head(self.backbone(x))

model = TinyDNACNNRegressor()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model

TinyDNACNNRegressor(
  (backbone): Sequential(
    (0): Conv1d(5, 32, kernel_size=(7,), stride=(1,), padding=(3,))
    (1): ReLU()
    (2): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (4): ReLU()
    (5): AdaptiveMaxPool1d(output_size=1)
  )
  (head): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=64, out_features=32, bias=True)
    (2): ReLU()
    (3): Linear(in_features=32, out_features=1, bias=True)
  )
)

## 6) Train for 10 cycles (easy to increase)

In [8]:
NUM_CYCLES = 10

def run_epoch(loader, train: bool):
    model.train(train)
    total_loss, total_mae, total = 0.0, 0.0, 0
    for xb, yb in loader:
        if train:
            optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_mae += (preds - yb).abs().sum().item()
        total += xb.size(0)
    return total_loss / total, total_mae / total

for cycle in range(1, NUM_CYCLES + 1):
    train_mse, train_mae = run_epoch(train_loader, train=True)
    val_mse, val_mae = run_epoch(val_loader, train=False)
    print(
        f"cycle={cycle:02d} "
        f"train_mse={train_mse:.4f} train_mae={train_mae:.4f} "
        f"val_mse={val_mse:.4f} val_mae={val_mae:.4f}"
    )

cycle=01 train_mse=9.1347 train_mae=1.0498 val_mse=446.5896 val_mae=9.1355
cycle=02 train_mse=9.0027 train_mae=0.9818 val_mse=445.2834 val_mae=9.0645
cycle=03 train_mse=8.8522 train_mae=0.9053 val_mse=443.6402 val_mae=8.9734
cycle=04 train_mse=8.7384 train_mae=0.8059 val_mse=441.6550 val_mae=8.8621
cycle=05 train_mse=8.5687 train_mae=0.6938 val_mse=439.5948 val_mae=8.7597
cycle=06 train_mse=8.3628 train_mae=0.6346 val_mse=437.5237 val_mae=8.7795
cycle=07 train_mse=8.3029 train_mae=0.6912 val_mse=435.2660 val_mae=8.8261
cycle=08 train_mse=8.1929 train_mae=0.7833 val_mse=433.1862 val_mae=8.8700
cycle=09 train_mse=8.1150 train_mae=0.8838 val_mse=431.4071 val_mae=8.9082
cycle=10 train_mse=8.0520 train_mae=0.9718 val_mse=430.0686 val_mae=8.9379


## 7) Quick test-set check

In [9]:
test_mse, test_mae = run_epoch(test_loader, train=False)
print(f"test_mse={test_mse:.4f} test_mae={test_mae:.4f}")

test_mse=0.3700 test_mae=0.5979
